# Federated Subset Scanning — full pipeline

One notebook, run top to bottom (**Runtime → Run all**, GPU runtime). Every number in the paper
should come from the **Results summary** cell at the end, which reads the JSON files the scripts write.

| Step | What | Time (approx.) |
|---|---|---|
| 0 | Setup: Drive, repo, packages | 2 min |
| 1 | Data: unzip PADS | 2 min |
| 2 | Base models (skip training if checkpoints exist) | fast / hours if retraining |
| 3 | Extract activations (layers 0–2, metadata) | ~10 min |
| 4 | DeepScan Methods A/B/C | ~30–60 min (FGSS) |
| 5 | Federated pseudo-site simulation (+ 5b site-count sweep) | ~10 min |
| 6 | Privacy: re-identification sweep | ~15 min |
| 7 | Privacy: membership inference + DP | ~10 min |
| 7b | Privacy v2: single volunteers, second attacker, A/B/C under DP | ~20–30 min |
| 8 | Compressed-input baseline | ~15 min |
| 8b | Figures (figures_v2/, figures_shared_doc/ incl. PCA) | ~5 min |
| 9 | Results summary (paper numbers) | instant |
| 10 | Backup results | 1 min |

Results go to `/content/drive/MyDrive/patchtst_checkpoints_nw5` (`RESULTS_DIR` in `scripts/config.py`).

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/federated-subset-scanning-pads'
if not os.path.exists(REPO):
    !git clone https://github.com/isaacmutuma/federated-subset-scanning-pads.git {REPO}
else:
    !git -C {REPO} fetch origin && git -C {REPO} reset --hard origin/main
%cd {REPO}

# Optional: pin an exact commit for a reproducible paper run, e.g. PIN_COMMIT = 'abc1234'
PIN_COMMIT = None
if PIN_COMMIT:
    !git -C {REPO} checkout {PIN_COMMIT}
!git -C {REPO} log -1 --oneline

In [ ]:
!pip install -q transformers accelerate scipy scikit-learn adversarial-robustness-toolbox
import torch, transformers, sklearn, numpy as np
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| transformers', transformers.__version__, '| sklearn', sklearn.__version__, '| numpy', np.__version__)

## 1. Data

In [ ]:
import zipfile
ZIP_PATH     = '/content/drive/MyDrive/pads-parkisons-dataset-folder/pads-parkinsons-disease-smartwatch-dataset-1.0.0.zip'
EXTRACT_PATH = '/content/pads_dataset'
if not os.path.exists(EXTRACT_PATH):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_PATH)
print(os.listdir(EXTRACT_PATH))

import sys
sys.path.insert(0, f'{REPO}/scripts')
import config as C
RESULTS_DIR = C.RESULTS_DIR
print('ENV:', C.ENV, '| TRAIN_WRIST:', C.TRAIN_WRIST, '| N_WINDOWS:', C.N_WINDOWS)

## 2. Base models
`train_patchtst.py` also writes `/content/processed/windows.npy` etc. (needed by later scripts) and
**skips training for any fold whose checkpoint already exists**. Set `RUN_BASELINES = True` to
re-run MiniROCKET / InceptionTime / BOSS for Table 1.

In [ ]:
!python scripts/train_patchtst.py 2>&1 | tail -10

RUN_BASELINES = True   # final run: also MiniROCKET, InceptionTime, BOSS (spec@90%sens)
if RUN_BASELINES:
    !python scripts/train_minirocket.py    2>&1 | tail -9
    !python scripts/train_inceptiontime.py 2>&1 | tail -9
    !python scripts/train_boss_svm.py      2>&1 | tail -9

## 3. Extract activations
Activations (layers 0–2, per-window metadata) were re-extracted on 2026-10-25 and are
cached on Drive; folds that already exist are skipped. Set `FORCE_REEXTRACT = True` only after
retraining or changing the extraction code. The printed **windows per subject** and
**skipped recordings** go in the paper.

In [ ]:
# Set True only after retraining checkpoints or changing extraction code
FORCE_REEXTRACT = False
!FORCE_REEXTRACT={int(FORCE_REEXTRACT)} python scripts/extract_activations.py 2>&1 | grep -vE "^  activations_" | grep -E "Windows|Skipped|per subject|HC=|FOLD|Already" | tail -40

## 4. DeepScan (Table 2, Table 3 DeepScan column)
Main run: held-out val HC reference, all tasks, both wrists, N=10 bins.
Variants (separate result files via `RUN_TAG`):
- `_hctrain` — old reference (HC the encoder trained on), to quantify the leakage effect
- `_relaxed_right` — Relaxed task, right wrist only (if the paper keeps that protocol)

In [ ]:
!python scripts/run_deepscan.py 2>&1 | grep -vE "^Fold [0-9] —|^fatal" | tail -60

In [ ]:
RUN_VARIANTS = True
if RUN_VARIANTS:
    !BACKGROUND=hc_train RUN_TAG=_hctrain python scripts/run_deepscan.py 2>&1 | grep -A6 "FINAL COMPARISON"
    !EVAL_TASK=relaxed EVAL_WRIST=right RUN_TAG=_relaxed_right python scripts/run_deepscan.py 2>&1 | grep -A6 "FINAL COMPARISON"

## 5. Federated pseudo-site simulation

In [ ]:
!python scripts/run_federated_sim.py 2>&1 | grep -E "FOLD|Central|SUMMARY|transmits|Error|Traceback"

### 5b. Site-count sweep (H1b)
63 HC per fold split evenly into 3/5/10/15/20 sites (21 down to 3-4 HC each). Federated - local AUC
with a paired bootstrap CI, test subjects pooled over folds. Rule fixed before the run: merging helps
at a site size if the CI lower bound is > 0. Writes `federated_sim_layer_2_sites.json`.

In [ ]:
!SITE_COUNTS=3,5,10,15,20 ASSIGN=balanced RUN_TAG=_sites python scripts/run_federated_sim.py > /content/fed_sites.log 2>&1; tail -n 12 /content/fed_sites.log

## 6. Privacy — re-identification sweep (figure: fig_reid_bins in 8b)

In [ ]:
!python scripts/privacy_analysis_bins.py 2>&1 | grep -E "Raw|N_BINS|ReID|DeepScan|Error|Traceback"

## 7. Privacy — membership inference on shared histograms (+ DP)
Attacks what a site actually transmits. Membership AUC 0.5 = no leakage.

In [ ]:
!python scripts/privacy_membership.py 2>&1 | grep -E "FOLD|Membership|Method A|SUMMARY|Error|Traceback" | tail -40

## 7b. Privacy v2 — single volunteers, second attacker, Methods B/C under DP
Histograms of 1/5/10/20 volunteers; attacker holds the contributed windows or other windows of the
same person; detection with A, B and C under the same noise. Output goes to a log file first.

In [ ]:
!python scripts/privacy_membership_v2.py > /content/privacy_v2.log 2>&1; tail -n 45 /content/privacy_v2.log

## 8. Compressed-input baseline

In [ ]:
!python scripts/compressed_input_baseline.py 2>&1 | tail -14

## 8b. Figures (single-axis, publication style)
ROC (Methods A/B/C), method AUC with 95% CI per configuration, membership inference vs site size,
re-identification vs bin count. Saved as PNG + PDF to `RESULTS_DIR/figures_v2/`.

In [ ]:
!python scripts/plot_results_v2.py
from IPython.display import Image, display
for f in ['fig_roc_layer2', 'fig_methods_by_config', 'fig_membership', 'fig_reid_bins']:
    p = f'{RESULTS_DIR}/figures_v2/{f}.png'
    if os.path.exists(p):
        display(Image(p, width=520))

# Shared-doc / paper figures 1-5 (PCA panels need analyze_layers.py; step 2 rebuilt windows.npy)
!python scripts/analyze_layers.py 2>&1 | tail -3
!python scripts/plot_shared_doc_figures.py
for f in ['fig1_system_overview', 'fig4_cross_validation', 'fig5_reid_vs_bins']:
    p = f'{RESULTS_DIR}/figures_shared_doc/{f}.png'
    if os.path.exists(p):
        display(Image(p, width=720))

## 9. Results summary — single source for paper numbers
Re-run this cell any time; it only reads JSON files. Missing files are reported, not fatal.

In [ ]:
import json, os.path as osp
import numpy as np

def load(name):
    p = osp.join(RESULTS_DIR, name)
    if not osp.exists(p):
        print(f'  [missing] {name}')
        return None
    with open(p) as f:
        return json.load(f)

def fmt(m, s):
    return f'{m:.3f} ± {s:.3f}'

print('=== Table 1: base models (subject-level AUC, 5-fold) ===')
for name, f in [('BOSS+SGD', 'boss_svm_results.json'), ('InceptionTime', 'inceptiontime_results.json'),
                ('PatchTST', 'patchtst_results.json'), ('MiniROCKET', 'minirocket_results.json')]:
    r = load(f)
    if r:
        a = r['aggregate']
        spec90 = (fmt(a['spec_at_90sens_mean'], a['spec_at_90sens_std'])
                  if 'spec_at_90sens_mean' in a else 'not in this file (re-run script)')
        print(f"  {name:<14} AUC {fmt(a['auc_mean'], a['auc_std'])}  spec@90%sens {spec90}")

for tag, title in [('', 'MAIN (val HC reference, all tasks)'), ('_hctrain', 'VARIANT: HC-train reference'),
                   ('_relaxed_right', 'VARIANT: Relaxed task, right wrist')]:
    print(f'\n=== Table 2/3 DeepScan — {title} ===')
    for layer in ['layer_0', 'layer_1', 'layer_2']:
        r = load(f'deepscan_results_{layer}{tag}.json')
        if not r:
            continue
        cfg = r['config']
        line = f"  {layer}  (bins={cfg['n_bins']}, bg={cfg['background']})"
        for m, lab in [('mean', 'A'), ('individ', 'B'), ('group', 'C')]:
            a = r.get(f'aggregate_{m}')
            if a:
                ci = a['pooled_bootstrap']
                line += (f"\n     {lab}: AUC {fmt(a['auc_mean'], a['auc_std'])}  "
                         f"pooled 95% CI [{ci['ci_low']:.3f}, {ci['ci_high']:.3f}]  "
                         f"spec@90%sens {a['spec_at_90sens_mean']:.3f}")
        if 'paired_A_vs_C' in r:
            p = r['paired_A_vs_C']
            line += f"\n     A − C: {p['diff']:+.3f}  95% CI [{p['ci_low']:+.3f}, {p['ci_high']:+.3f}]  p={p['p']:.3f}"
        line += f"\n     FGSS failures: {r.get('fgss_failures')}  commit {cfg['git_commit'][:7]}"
        print(line)

print('\n=== Federated simulation ===')
r = load('federated_sim_layer_2.json')
if r:
    s = r['summary']
    print(f"  central {fmt(*s['central_auc'])} | federated {fmt(*s['federated_auc_mean'])} | "
          f"local-only {fmt(*s['local_auc_mean'])}  ({r['config']['n_sites']} sites, "
          f"{r['bytes_per_site_counts']} bytes/site)")
r = load('federated_sim_layer_2_sites.json')
if r:
    print('  site-count sweep (balanced): sites | HC/site | local-only | federated - local [95% CI]')
    for n, o in r['by_n_sites'].items():
        sm, g = o['summary'], o['pooled_gain_median']
        print(f"    {n:>2} | {sm['site_hc_subjects_min_max'][0]}-{sm['site_hc_subjects_min_max'][1]:<2} | "
              f"{fmt(*sm['local_auc_mean'])} | {g['diff']:+.3f} [{g['ci_low']:+.3f}, {g['ci_high']:+.3f}]"
              f"{'  merging helps' if g['merging_helps'] else ''}")

print('\n=== Privacy: re-identification (Table 4, Fig. 2) ===')
r = load('privacy_bins_sweep.json')
if r:
    raw = r['raw_reid']
    print(f"  attack subjects/fold {r['n_attack_subjects_per_fold']}  chance Top-1 {r['chance_top1']:.2%}  "
          f"Top-5 {r['chance_top5']:.2%}")
    print(f"  raw activations: Top-1 {raw['top1_mean']:.2%} ± {raw['top1_std']:.2%}  "
          f"Top-5 {raw['top5_mean']:.2%}")
    for b, v in r['per_bin'].items():
        print(f"  N={b:>3}: p-values Top-1 {v['reid_top1_mean']:.2%}  Top-5 {v['reid_top5_mean']:.2%} | "
              f"quantised raw Top-1 {v['quant_reid_top1_mean']:.2%} | "
              f"Method A AUC {fmt(v['deepscan_auc_mean'], v['deepscan_auc_std'])}")

print('\n=== Privacy: membership inference on shared histograms ===')
r = load('privacy_membership_layer_2.json')
if r:
    for k, v in r['summary']['membership'].items():
        print(f"  membership AUC {k:<18} {v:.3f}")
    for k, v in r['summary']['utility'].items():
        print(f"  utility (Method A AUC) {k:<10} {v:.3f}")

print('\n=== Privacy v2: membership by volunteers per histogram (contributed / other windows) ===')
r = load('privacy_membership_v2_layer_2.json')
if r:
    m, cfg = r['summary']['membership'], r['config']
    for k in cfg['site_sizes']:
        print(f"  {k:>2} volunteers  " + "  ".join(
            f"eps={str(e):>4}: {m[f'exact_size{k}_eps{e}']:.3f} / {m[f'fresh_size{k}_eps{e}']:.3f}"
            for e in cfg['epsilons']))
    for e in cfg['epsilons']:
        print(f"  detection eps={str(e):>4}: " + "  ".join(
            f"{x} AUC {r['summary']['detection'][f'{x}_eps{e}']['auc']:.3f}" for x in 'ABC'))
    print(f"  scan failures {r['scan_failures']}  commit {cfg['git_commit'][:7]}")

print('\n=== Compressed-input baseline ===')
r = load('compressed_input_baseline.json')
if r:
    for k, v in r['summary'].items():
        print(f"  {k:<48} {fmt(v['mean'], v['std'])}")

## 10. Backup

In [ ]:
import shutil, datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
jsons = [f for f in os.listdir(RESULTS_DIR) if f.endswith(('.json', '.png', '.pdf'))]
bdir = f'/content/drive/MyDrive/results_backup_{stamp}'
os.makedirs(bdir, exist_ok=True)
for f in jsons:
    shutil.copy(osp.join(RESULTS_DIR, f), bdir)
print(f'Copied {len(jsons)} result files to {bdir}')